In [ ]:
from transformers import GPT2Tokenizer
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn

In [ ]:
data = np.memmap(
    "sql_tokens.bin",
    dtype=np.uint16,
    mode="r"
)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
len(data)

57812765

In [ ]:
from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

vocab_size = tokenizer.vocab_size

tokenizer.model_max_length = int(1e9)

def encode(text):
    return tokenizer.encode(text, add_special_tokens=False)

def decode(tokens):
    return tokenizer.decode(tokens)

In [ ]:
# instruction_lengths = []
# output_lengths = []
# example_lengths = []

# for i in range(len(data)):
#     instruction = str(dataOld['instruction'].iloc[i])
#     output = str(dataOld['output'].iloc[i])

#     instruction_lengths.append(len(encode(instruction)))
#     output_lengths.append(len(encode(output)))

#     example = (
#         "### Instruction:\n\n"
#         + instruction
#         + "\n### Output:\n\n"
#         + output
#         + "\n### End"
#     )

#     example_lengths.append(len(encode(example)))

# print("Instruction:")
# print("  Average:", sum(instruction_lengths) / len(instruction_lengths))
# print("  Max:", max(instruction_lengths))

# print("\nOutput:")
# print("  Average:", sum(output_lengths) / len(output_lengths))
# print("  Max:", max(output_lengths))

# print("\nComplete example:")
# print("  Average:", sum(example_lengths) / len(example_lengths))
# print("  Max:", max(example_lengths))

In [ ]:
## Context Size and Chunking - Block size is the context length, ie how many past tokens the model sees, and batch_size is how many parallel operations run in the model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            np.array(data[i:i+block_size], dtype=np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            np.array(data[i+1:i+block_size+1], dtype=np.int64)
        )
        for i in ix
    ])

    x, y = x.to(device), y.to(device)

    return x, y

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
batch_size = 16
block_size = 512
max_iters = 3000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

n_embd = 288
n_head = 8
n_layer = 10
dropout = 0.0

In [ ]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Positional embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head=n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layer norm
        self.ln_f = nn.LayerNorm(n_embd)

        # Language-model head
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # idx and targets are both (B,T) tensors of integers

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)
        # (B,T,C)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )
        # (T,C)

        # Add token + positional embeddings
        x = tok_emb + pos_emb
        # (B,T,C)

        # Transformer
        x = self.blocks(x)
        # (B,T,C)

        # Final layer norm
        x = self.ln_f(x)
        # (B,T,C)

        # Convert to vocabulary logits
        logits = self.lm_head(x)
        # (B,T,vocab_size)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, stop_text=None):
      for _ in range(max_new_tokens):

          # Crop context to block size
          idx_cond = idx[:, -block_size:]

          # Get predictions
          logits, loss = self(idx_cond)

          # Focus only on the last time step
          logits = logits[:, -1, :]

          # Convert logits to probabilities
          probs = F.softmax(logits, dim=-1)

          # Sample next token
          idx_next = torch.argmax(logits, dim=-1, keepdim=True)

          # Append token
          idx = torch.cat((idx, idx_next), dim=1)

          # Stop if ### End appears
          if stop_text is not None:
              generated_text = decode(idx[0].tolist())

              if stop_text in generated_text:
                  break

      # MUST be outside the for loop
      return idx

In [ ]:
# old_state = m.state_dict()

# model = GPTLanguageModel()
# m = model.to(device)

# m.load_state_dict(old_state)

In [ ]:
model = GPTLanguageModel()
m = model.to(device)

# checkpoint = torch.load(
#     "gpt_python_checkpoint.pth",
#     map_location=device,
#     weights_only=False
# )

# m.load_state_dict(checkpoint["model_state_dict"])

print(
    sum(p.numel() for p in m.parameters()) / 1e6,
    'M parameters'
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

39.128401 M parameters


In [ ]:
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:

        losses = estimate_loss()

        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

step 0: train loss 1.7992, val loss 1.3927
step 100: train loss 1.7014, val loss 1.2749
step 200: train loss 1.6061, val loss 1.2189
step 300: train loss 1.5326, val loss 1.1397
step 400: train loss 1.4661, val loss 1.0881
step 500: train loss 1.4195, val loss 1.0382
step 600: train loss 1.3695, val loss 0.9964
step 700: train loss 1.3148, val loss 0.9685
step 800: train loss 1.2853, val loss 0.9277
step 900: train loss 1.2430, val loss 0.8919
step 1000: train loss 1.2114, val loss 0.8626
step 1100: train loss 1.1512, val loss 0.8379
step 1200: train loss 1.1129, val loss 0.8083
step 1300: train loss 1.0839, val loss 0.7826
step 1400: train loss 1.0619, val loss 0.7532
step 1500: train loss 1.0185, val loss 0.7239
step 1600: train loss 0.9727, val loss 0.7109
step 1700: train loss 0.9635, val loss 0.6861
step 1800: train loss 0.9510, val loss 0.6645
step 1900: train loss 0.9043, val loss 0.6409
step 2000: train loss 0.8775, val loss 0.6244
step 2100: train loss 0.8665, val loss 0.5959


In [ ]:
# Save final checkpoint
checkpoint_path = "gpt_python_checkpoint.pth"

torch.save({
    "model_state_dict": m.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iter": max_iters,
}, checkpoint_path)

print(f"Saved checkpoint: {checkpoint_path}")

# Automatically download to your computer
from google.colab import files
files.download(checkpoint_path)

Saved checkpoint: gpt_python_checkpoint.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# m.eval()

prompt = "### Schema CREATE TABLE users (id INT, name TEXT, age INT ); ### Question Show all users older than 25. ### SQL"

context = torch.tensor(
    [encode(prompt)],
    dtype=torch.long,
    device=device
)

generated = m.generate(
    context,
    max_new_tokens=200,
    stop_text="### End"
)

print(decode(generated[0].tolist()))

### Schema CREATE TABLE users (id INT, name TEXT, age INT ); ### Question Show all users older than 25. ### SQLor_id, gender TEXT); INSERT INTO users (id, name, age, gender) VALUES (1, 'John Doe', 35, 'female'), (2, 'Jane Smith', 40, 'male');
SELECT COUNT(*) FROM users WHERE gender = 'female';
What is the total number of visitors who attended the United States in the last 30 days?
CREATE TABLE Visitors (id INT, name VARCHAR(255), country VARCHAR(255), visit_date DATE); INSERT INTO Visitors (id, name, country, visit_date) VALUES (1, 'Alice', 'USA', '2022-01-01'), (2, 'Bob', 'Canada', '2022-01-01'), (3, 'Charlie', 'Mexico', '2022-01-01'), (4, 'David', 'Mexico', '2022-01-01'), (5, 'David', '


In [ ]:
from google.colab import drive
import torch
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths
checkpoint_path = "/content/drive/MyDrive/gpt_python_checkpoint.pth"
tokens_src = "tokens.bin"
tokens_dst = "/content/drive/MyDrive/tokens.bin"

# Save model checkpoint
torch.save({
    "model_state_dict": m.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iter": max_iters,
}, checkpoint_path)

print(f"Saved checkpoint: {checkpoint_path}")

# Save tokenized dataset
if os.path.exists(tokens_src):
    shutil.copy(tokens_src, tokens_dst)
    print(f"Saved tokens: {tokens_dst}")
else:
    print("tokens.bin not found")

In [ ]:
print(decode(encode("Create a table as users, with fields UserID, Name, and Email.")))

xb, yb = get_batch("train")
print(decode(xb[0][:200].tolist()))

In [ ]:
print(decode(data[:300].tolist()))